# Bank marketing, done carefully

The same question with a decision-time contract, a locked split, fold-local preprocessing, a model screen and honest metrics. The copilot should find nothing above `low` here.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
ROOT = Path.cwd()
while not (ROOT / "external_data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
X = pd.read_parquet(ROOT / "external_data/bank_marketing/X.parquet")
y = pd.read_parquet(ROOT / "external_data/bank_marketing/y.parquet")["target"]
# Decision moment: just before the call. Call length is only known afterwards.
X = X.drop(columns=["duration"])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
candidates = {
    "dummy": DummyClassifier(strategy="prior"),
    "logistic": make_pipeline(SimpleImputer(), StandardScaler(), LogisticRegression(max_iter=2000)),
    "hist_gb": HistGradientBoostingClassifier(random_state=42),
}
cv_scores = {name: cross_val_score(est, X_train, y_train, cv=cv, scoring="average_precision") for name, est in candidates.items()}
{name: (s.mean().round(4), s.std().round(4)) for name, s in cv_scores.items()}

In [ ]:
final = candidates["hist_gb"].fit(X_train, y_train)
p = final.predict_proba(X_test)[:, 1]
print("ROC-AUC", roc_auc_score(y_test, p), "AP", average_precision_score(y_test, p), "Brier", brier_score_loss(y_test, p))